# PID loop simulator
## Run a closed loop PID simulator to allow testing different options.
Craig Lage 08-Apr-26

In [ ]:
import random
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId
from lsst.ts.ofc import OFC, OFCData, StateEstimator, SensitivityMatrix
import galsim

## The cell below gets data that allows testing.  May not be needed long term.

In [ ]:
day_obs = 20260404

# Load the nightly table.
# The parquet file was created with: 
# https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb
parquet_file = f"/home/c/cslage/u/MTAOS/times_square_reports/nightly_aos_table_{day_obs}.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)
    

## Build ofc, state estimator and sensitivity matrix

In [ ]:
ofc_data = OFCData(
    name="lsst",
    config_dir="/home/c/cslage/WORK/ts_config_mttcs/MTAOS/ofc"
)

ofc = OFC(
    ofc_data=ofc_data,
)

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)

new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)
new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

ofc.set_truncation_index(12)
ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

sensor_ids=np.array([191, 195, 199, 203])
corner_detnames = [  # extra-focal
    "R00_SW0", "R04_SW0",
    "R40_SW0", "R44_SW0",
]
field_angles_CCS = [
    ofc_data.sample_points[det] for det in corner_detnames
]

state_estimator = StateEstimator(ofc_data)
sens_mat = state_estimator.get_sensitivity_matrix(field_angles_CCS, 0.0)

In [ ]:
ofc.controller.integral

## Calculate correction from a set of zernikes

In [ ]:
def get_visit_dof(ofc, zernikes, Kp, Ki, Kd, rotation_angle=0.0, filter_name='i'):
    sensor_ids = [191, 195, 199, 203]
    ofc.controller.kp = Kp
    ofc.controller.ki = Ki
    ofc.controller.kd = Kd
    ofc.controller.reset_history()
    m2hex_1, camhex_1, m1m3_, m2_1 = ofc.calculate_corrections(zernikes, sensor_ids, filter_name,
                                                   rotation_angle, subtract_intrinsics=False,
                                                  control_vmodes=False)
    visit_dof = ofc.lv_dof[indices]
    return visit_dof

In [ ]:
def get_new_zernikes(zernikes, visit_dof, sens_mat):
    zernikes_change = sens_mat @ visit_dof
    zernikes_change = zernikes_change.reshape(4,21)
    zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
    zernikes_change = np.insert(zernikes_change, obj=16, values=0, axis=1)
    return zernikes - zernikes_change
    

In [ ]:
noise_matrix = 0.08 / getPsfGradPerZernike(jmax=26)
no_noise = np.zeros(23)
def run_PID_loop(ofc, initial_zernikes, Kp, Ki, Kd, rotation_angle=0.0, filter_name='i', num_steps=10, noise_matrix=no_noise):
    loop_zernikes_fwhm = []
    zernikes = initial_zernikes
    loop_zernikes_fwhm.append(convertZernikesToPsfWidth(zernikes))
    for i in range(num_steps):
        visit_dof = get_visit_dof(ofc, zernikes, Kp, Ki, Kd, rotation_angle=0.0, filter_name='i')
        zernikes = get_new_zernikes(zernikes, visit_dof, sens_mat)
        for i in range(4):
            for j in range(23):
                zernikes[i, j] += random.gauss(0.0, noise_matrix[j])
        loop_zernikes_fwhm.append(convertZernikesToPsfWidth(zernikes))
    return np.array(loop_zernikes_fwhm)

In [ ]:
def getPsfGradPerZernike(
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
    jmax: int = 22,
) -> np.ndarray:
    """Get the gradient of the PSF FWHM with respect to each Zernike.

    This function takes no positional arguments. All parameters must be passed
    by name (see the list of parameters below).

    Parameters
    ----------
    diameter : float, optional
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float, optional
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int, optional
        The minimum Noll index, inclusive. Must be >= 0. (the default is 4)
    jmax : int, optional
        The max Zernike Noll index, inclusive. Must be >= jmin.
        (the default is 22.)

    Returns
    -------
    np.ndarray
        Gradient of the PSF FWHM with respect to the corresponding Zernike.
        Units are arcsec / micron.

    Raises
    ------
    ValueError
        If jmin is negative or jmax is less than jmin
    """
    # Check jmin and jmax
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")
    if jmax < jmin:
        raise ValueError("jmax must be greater than jmin.")

    # Calculate the conversion factors
    conversion_factors = np.zeros(jmax + 1)
    for i in range(jmin, jmax + 1):
        # Set coefficients for this Noll index: coefs = [0, 0, ..., 1]
        # Note the first coefficient is Noll index 0, which does not exist and
        # is therefore always ignored by galsim
        coefs = [0] * i + [1]

        # Create the Zernike polynomial with these coefficients
        R_outer = diameter / 2
        R_inner = R_outer * obscuration
        Z = galsim.zernike.Zernike(coefs, R_outer=R_outer, R_inner=R_inner)

        # We can calculate the size of the PSF from the RMS of the gradient of
        # the wavefront. The gradient of the wavefront perturbs photon paths.
        # The RMS quantifies the size of the collective perturbation.
        # If we expand the wavefront gradient in another series of Zernike
        # polynomials, we can exploit the orthonormality of the Zernikes to
        # calculate the RMS from the Zernike coefficients.
        rms_tilt = np.sqrt(np.sum(Z.gradX.coef**2 + Z.gradY.coef**2) / 2)

        # Convert to arcsec per micron
        rms_tilt = np.rad2deg(rms_tilt * 1e-6) * 3600

        # Convert rms -> fwhm
        fwhm_tilt = 2 * np.sqrt(2 * np.log(2)) * rms_tilt

        # Save this conversion factor
        conversion_factors[i] = fwhm_tilt

    return conversion_factors[jmin:]



def convertZernikesToPsfWidth(
    zernikes: np.ndarray,
    diameter: float = 8.36,
    obscuration: float = 0.612,
    jmin: int = 4,
) -> np.ndarray:
    """Convert Zernike amplitudes to quadrature contribution to the PSF FWHM.

    Parameters
    ----------
    zernikes : np.ndarray
        Zernike amplitudes (in microns), starting with Noll index `jmin`.
        Either a 1D array of zernike amplitudes, or a 2D array, where each row
        corresponds to a different set of amplitudes.
    diameter : float
        The diameter of the telescope aperture, in meters.
        (the default, 8.36, corresponds to the LSST primary mirror)
    obscuration : float
        Central obscuration of telescope aperture (i.e. R_outer / R_inner).
        (the default, 0.612, corresponds to the LSST primary mirror)
    jmin : int
        The minimum Zernike Noll index, inclusive. Must be >= 0. The
        max Noll index is inferred from `jmin` and the length of `zernikes`.
        (the default is 4, which ignores piston, x & y offsets, and tilt.)

    Returns
    -------
    dFWHM: np.ndarray
        Quadrature contribution of each Zernike vector to the PSF FWHM
        (in arcseconds).

    Notes
    -----
    Converting Zernike amplitudes to their quadrature contributions to the PSF
    FWHM allows for easier physical interpretation of Zernike amplitudes and
    the performance of the AOS system.

    For example, image we have a true set of zernikes, [Z4, Z5, Z6], such that
    ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.2, 0.3] arcsecs.
    These Zernike perturbations increase the PSF FWHM by
    sqrt[(0.1)^2 + (-0.2)^2 + (0.3)^2] ~ 0.37 arcsecs.

    If the AOS perfectly corrects for these perturbations, the PSF FWHM will
    not increase in size. However, imagine the AOS estimates zernikes, such
    that ConvertZernikesToPsfWidth([Z4, Z5, Z6]) = [0.1, -0.3, 0.4] arcsecs.
    These estimated Zernikes, do not exactly match the true Zernikes above.
    Therefore, the post-correction PSF will still be degraded with respect to
    the optimal PSF. In particular, the PSF FWHM will be increased by
    sqrt[(0.1 - 0.1)^2 + (-0.2 - (-0.3))^2 + (0.3 - 0.4)^2] ~ 0.14 arcsecs.

    This conversion depends on a linear approximation that begins to break down
    for RSS(dFWHM) > 0.20 arcsecs. Beyond this point, the approximation tends
    to overestimate the PSF degradation. In other words, if
    sqrt(sum( dFWHM^2 )) > 0.20 arcsec, it is likely that dFWHM is
    over-estimated. However, the point beyond which this breakdown begins
    (and whether the approximation over- or under-estimates dFWHM) can change,
    depending on which Zernikes have large amplitudes. In general, if you have
    large Zernike amplitudes, proceed with caution!
    Note that if the amplitudes Z_est and Z_true are large, this is okay, as
    long as |Z_est - Z_true| is small.

    For a notebook demonstrating where the approximation breaks down:
    https://gist.github.com/jfcrenshaw/24056516cfa3ce0237e39507674a43e1

    Raises
    ------
    ValueError
        If jmin is negative
    """
    # Check jmin
    if jmin < 0:
        raise ValueError("jmin cannot be negative.")

    # Calculate jmax from jmin and the length of the zernike array
    jmax = jmin + np.array(zernikes).shape[-1] - 1

    # Calculate the conversion factors for each zernike
    conversion_factors = getPsfGradPerZernike(
        jmin=jmin,
        jmax=jmax,
        diameter=diameter,
        obscuration=obscuration,
    )

    # Convert the Zernike amplitudes from microns to their quadrature
    # contribution to the PSF FWHM
    dFWHM = conversion_factors * zernikes

    return dFWHM



# Start with an initial set of Zernikes from real data

In [ ]:
expId = 2026040400279
seqNum = int(expId - 1E5 * day_obs)
this_table = table[table['seq'] == seqNum]
zernikes = np.zeros([4,23])
zernikes[0,:] = this_table['zk_deviation_R00'].values[0]
zernikes[1,:] = this_table['zk_deviation_R04'].values[0]
zernikes[2,:] = this_table['zk_deviation_R40'].values[0]
zernikes[3,:] = this_table['zk_deviation_R44'].values[0]

In [ ]:
num_steps = 20
Kp = 0.3
Ki = 0.0
Kd = 0.0
loop_zernikes = run_PID_loop(ofc, zernikes, Kp, Ki, Kd, rotation_angle=0.0, num_steps=num_steps, noise_matrix=noise_matrix)

In [ ]:
zk_groups = [[0], [11 - 4], [1, 2], [3,4], [5, 6], [22]]
zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10', 'Z22']
fig = plt.figure(figsize=(10,10))
  
gs = gridspec.GridSpec(
nrows=6, ncols=1,
height_ratios=[1]*6,
hspace=0.0,
wspace=0.26,
top=0.95,
bottom=0.05  # ends halfway down
)

xaxis = np.arange(num_steps + 1) 
axes = [fig.add_subplot(gs[i, 0]) for i in range(6)]
for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
    zk_labels = zk_group_labels[id_group]
    for zk_idx, i in enumerate(zk_group):
        ymin =-0.5; ymax = 0.5
    for zk_idx, i in enumerate(zk_group):
        # This loop does the scatter plots
        ymin_microns = ymin / getPsfGradPerZernike(jmax=26)[i]
        ymax_microns = ymax / getPsfGradPerZernike(jmax=26)[i]
        if len(zk_group) == 1:
            zk_label = ''
        if len(zk_group) == 2:
            zk_label = zk_labels.split('/')[zk_idx].strip()
        vals = np.mean(loop_zernikes[:,:,i], axis=1)
        color = 'tab:blue' if zk_idx == 0 else 'tab:orange' if zk_idx == 1 else None
        clip_vals = np.clip(vals, ymin, ymax)
        ax.scatter(xaxis, clip_vals, s=20, color=color, label=zk_label)
    ax.set_ylabel(f"{zk_group_labels[id_group]}\n[arcsec]")
    if zk_group_labels[id_group] == 'Z4':
        # This line is showing the recently added focus offset
        ax.axhline(-0.15, ls='--', color='green')
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("PID step")
    ax2 = ax.twinx()
    ax2.set_ylim(ymin_microns, ymax_microns)
    ax2.set_ylabel("WF deviation\n[um]")
    leg1 = ax.legend(bbox_to_anchor=(-0.20, 0.5), loc='upper left',
                     ncol=1, markerscale=2, frameon=False)
my_suptitle = fig.suptitle(f"Simulated PID loop start={expId}\nKp={Kp}, Ki={Ki}, Kd={Kd}, with noise, every image", y=1.02)

fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/PID_Simulator_With_Noise{expId}.png", 
            bbox_inches='tight', pad_inches=1.2, bbox_extra_artists=[my_suptitle])